# 01 · Data preparation

This notebook takes the eight datasets from their original raw files to the ready-to-use files in `data/processed/`. Every dataset goes through the same five steps, in this order:

| Step | What happens |
|---|---|
| 1. Load | Read the raw files exactly as downloaded (UCI, Kaggle, OpenML). . |
| 2. Audit | Inspect every raw dataset with the same checks. Results are printed, not saved. |
| 3. Clean | Apply the written cleaning rule of each dataset. |
| 4. Prepare | One common pipeline: 350 rows, 280 / 70 split, filters fitted on training rows only, scaling to [0, π]. |
| 5. Check and save | Compare with the files in `data/processed/`. Files are written only when `SAVE = True`. |

**Rules that hold everywhere**

- The raw files in `data/raw/` are never changed.
- The 70 test rows of each dataset are split off before any fitting and stay sealed until the final evaluation.
- Every fitted step (imputation, correlation filter, feature selection, scaling) uses the 280 training rows only.

## The eight datasets

| Dataset | Source | Task (label = 1) | Qubits | Why it is in the study |
|---|---|---|---|---|
| German Credit | UCI (numeric version) | good credit risk (70 %) | 14 | many category-coded features |
| Concrete | UCI | strong concrete (above the training median) | 8 | continuous recipe measurements |
| Water Potability | Kaggle | water is drinkable (39 %) | 9 | real missing values, very weak signal |
| Steel Plates Faults | UCI | fault type "Other_Faults" (35 %) | 14 | industrial measurements, many features |
| Phoneme | OpenML 1489 | the smaller of two sound classes (29 %) | 5 | fewest qubits, non-linear |
| MAGIC Telescope | OpenML 1120 | hadron shower, not gamma ray (35 %) | 8 | strong but non-linear signal, skewed features |
| Bank Marketing | OpenML 1461 | customer opened a deposit (12 %) | 14 | rare positives, text categories |
| Electricity | OpenML 151 | price goes up (42 %) | 6 | time-of-day and weekday features |

One qubit is used per feature, so the number of features after preparation is the number of qubits.

## 0 · Setup

`SAVE = False` means the notebook builds everything in memory and only compares it with the existing files. Set it to `True` only to rebuild `data/processed/` on purpose; it writes the same 16 file names and nothing else.

In [1]:
import warnings
import numpy as np
import pandas as pd
from scipy.stats import skew
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from paths import RAW, PROCESSED

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

SAVE = False          # True = write the 16 ready files to data/processed/

SEED = 42             # same seed for sampling, splitting and feature selection
N_ROWS = 350          # rows kept per dataset (simulation cost)
TEST_SHARE = 0.20     # 280 training rows / 70 test rows
CORR_LIMIT = 0.85     # drop one feature of every pair above this |r|
QUBIT_CAP = 14        # at most 14 features = 14 qubits

DATASETS = ["german_credit", "concrete", "water_potability", "steel_faults",
            "phoneme", "magic_telescope", "bank_marketing", "electricity"]

## 1 · Load the raw data

Each loader returns the raw table with the label column renamed to `target` and nothing else changed (Bank Marketing also gets its real column names; OpenML stores them as V1–V16). The OpenML datasets are read from the local copy in `data/raw/openml/`.

In [2]:
OPENML_IDS = {"phoneme": 1489, "magic_telescope": 1120, "bank_marketing": 1461, "electricity": 151}
BANK_NAMES = {"V1": "age", "V2": "job", "V3": "marital", "V4": "education", "V5": "default", "V6": "balance",
              "V7": "housing", "V8": "loan", "V9": "contact", "V10": "day", "V11": "month", "V12": "duration",
              "V13": "campaign", "V14": "pdays", "V15": "previous", "V16": "poutcome"}


def load_openml(name):
    b = fetch_openml(data_id=OPENML_IDS[name], as_frame=True, parser="auto", data_home=str(RAW / "openml"))
    df = b.frame.rename(columns={b.target.name: "target"})#renames the target column
    df.columns = [c.rstrip(":") for c in df.columns]#emoves trailing colons (:) from column names, if present.
    return df.rename(columns=BANK_NAMES) if name == "bank_marketing" else df   # OpenML stores Bank as V1-V16


def load_german():
    df = pd.read_csv(RAW / "german_credit.csv", sep=r"\s+", header=None)
    df.columns = [f"f{i}" for i in df.columns[:-1]] + ["target"]
    return df


def load_concrete():
    df = pd.read_csv(RAW / "Concrete_Data.csv", encoding="latin-1")
    df.columns = df.columns.str.encode("ascii", "ignore").str.decode("ascii").str.strip()
    return df.rename(columns={df.columns[-1]: "target"})


def load_water():
    return pd.read_csv(RAW / "water_potability.csv").rename(columns={"Potability": "target"})


def load_steel():
    names = [c.strip() for c in open(RAW / "Faults27x7_var").read().splitlines() if c.strip()]
    return pd.read_csv(RAW / "Faults.NNA", sep="\t", header=None, names=names)


LOADERS = {"german_credit": load_german, "concrete": load_concrete, "water_potability": load_water,
           "steel_faults": load_steel, **{n: (lambda n=n: load_openml(n)) for n in OPENML_IDS}}

raw = {name: LOADERS[name]() for name in DATASETS}#Load all eight datasets
pd.DataFrame({name: {"rows": len(df), "columns": df.shape[1]} for name, df in raw.items()}).T

,rows,columns
german_credit,1000,25
concrete,1030,9
water_potability,3276,10
steel_faults,1941,34
phoneme,5404,6
magic_telescope,19020,11
bank_marketing,45211,17
electricity,45312,9


## 2 · Audit the raw data

The same checks for all eight datasets, on the full raw data before any cleaning:

- **size and balance** — rows, features, share of the smaller class;
- **missing values** — empty cells, and text codes such as "unknown";
- **repeated rows** — exact copies, and rows with identical features but a different label;
- **suspicious columns** — constant columns, ID-like columns, one value filling most of a column (a filler), code values such as −1 or 999;
- **shape of the values** — strongly skewed columns and extreme outliers;
- **leakage** — a single feature that predicts the label almost perfectly on its own.

Steel Plates stores its label as seven one-hot fault columns, so its label for the audit is `Other_Faults`.

In [3]:
CODES = [-1, -9, -99, -999, 99, 999, 9999]
TEXT_MISSING = ["unknown", "?", "na", "none", ""]


def audit_label(name, df):
    if name == "steel_faults":
        return df["Other_Faults"].astype(int)
    y = df["target"]
    if pd.api.types.is_numeric_dtype(y) and y.nunique() > 10:      # Concrete: strength in MPa
        return (y >= y.median()).astype(int)
    return (y == y.value_counts().idxmin()).astype(int)


def audit(name, df):
    y = audit_label(name, df)
    drop = ["target"] if "target" in df else ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]
    X = df.drop(columns=drop)
    num = X.select_dtypes("number")
    txt = X.drop(columns=num.columns)
    flags = []

    same_x = pd.DataFrame({"key": [tuple(r) for r in X.astype(str).values], "y": y}).groupby("key").y.nunique()
    conflicts = int((same_x > 1).sum())

    filler, codes, skewed, outliers, ids = [], [], [], [], []
    for c in num.columns:
        s = num[c].dropna()
        top = s.value_counts(normalize=True).iloc[0]
        if top > 0.30 and s.nunique() > 10:
            filler.append(f"{c} ({top:.0%})")
        share = s.isin(CODES).mean()
        if share > 0.01:
            codes.append(f"{c} ({share:.0%})")
        if abs(skew(s)) > 2:
            skewed.append(c)
        q1, q3 = s.quantile([0.25, 0.75])
        if q3 > q1 and ((s < q1 - 3 * (q3 - q1)) | (s > q3 + 3 * (q3 - q1))).mean() > 0.02:
            outliers.append(c)
        if c.lower() in ("id", "index", "date") or (s.is_monotonic_increasing and s.nunique() > 0.5 * len(s)):
            ids.append(c)

    unknown = [f"{c} ({txt[c].astype(str).str.lower().isin(TEXT_MISSING).mean():.0%})" for c in txt.columns
               if txt[c].astype(str).str.lower().isin(TEXT_MISSING).any()]

    best_c, best_auc = None, 0.5
    for c in X.columns:
        score = X[c].astype(float) if c in num else X[c].astype(str).map(y.groupby(X[c].astype(str)).mean())
        score = score.fillna(score.median())
        if score.nunique() > 1:
            a = roc_auc_score(y, score)
            if max(a, 1 - a) > best_auc:
                best_c, best_auc = c, max(a, 1 - a)

    row = {"rows": len(df), "features": X.shape[1], "text features": txt.shape[1],
           "smaller class": f"{min(y.mean(), 1 - y.mean()):.0%}",
           "missing cells": f"{X.isna().mean().mean():.1%}",
           "duplicate rows": int(df.duplicated().sum()),
           "same features, other label": conflicts,
           "constant columns": int((X.nunique() <= 1).sum()),
           "few-valued columns (<= 10)": int((X.nunique() <= 10).sum()),
           "skewed columns (|skew| > 2)": len(skewed),
           "strongest single feature": f"{best_c} ({best_auc:.2f})"}

    if X.isna().any().any():  flags.append("missing cells: " + ", ".join(f"{c} ({v:.0%})" for c, v in X.isna().mean().items() if v > 0))
    if unknown:               flags.append("text missing codes: " + ", ".join(unknown))
    if row["duplicate rows"]: flags.append(f"{row['duplicate rows']} exact duplicate rows")
    if conflicts:             flags.append(f"{conflicts} groups of identical features with a different label")
    if ids:                   flags.append("ID / time-index columns: " + ", ".join(ids))
    if filler:                flags.append("one value fills much of a column: " + ", ".join(filler))
    if codes:                 flags.append("code values (-1, 999, ...): " + ", ".join(codes))
    if outliers:              flags.append("extreme outliers (> 2 % beyond 3 IQR): " + ", ".join(outliers))
    if best_auc >= 0.80:      flags.append(f"possible leakage: {best_c} alone predicts the label (AUC {best_auc:.2f})")
    return row, flags


audit_rows, audit_flags = {}, {}
for name in DATASETS:
    audit_rows[name], audit_flags[name] = audit(name, raw[name])

pd.DataFrame(audit_rows)

,german_credit,concrete,water_potability,steel_faults,phoneme,magic_telescope,bank_marketing,electricity
rows,1000,1030,3276,1941,5404,19020,45211,45312
features,24,8,9,27,5,10,16,8
text features,0,0,0,0,0,0,9,1
smaller class,30%,50%,39%,35%,29%,35%,12%,42%
missing cells,0.0%,0.0%,4.9%,0.0%,0.0%,0.0%,0.0%,0.0%
duplicate rows,0,25,0,0,9,115,0,0
"same features, other label",0,2,0,0,0,0,0,0
constant columns,0,0,0,0,0,0,0,0
few-valued columns (<= 10),21,0,0,3,0,0,7,1
skewed columns (|skew| > 2),5,1,0,8,0,2,5,2


In [4]:
for name in DATASETS:
    print(name.upper())
    for f in audit_flags[name] or ["no flags"]:
        print("   -", f)

GERMAN_CREDIT
   - extreme outliers (> 2 % beyond 3 IQR): f3
CONCRETE
   - 25 exact duplicate rows
   - 2 groups of identical features with a different label
   - one value fills much of a column: Blast Furnace Slag (component 2)(kg in a m^3 mixture) (46%), Fly Ash (component 3)(kg in a m^3 mixture) (55%), Superplasticizer (component 5)(kg in a m^3 mixture) (37%), Age (day) (41%)
   - extreme outliers (> 2 % beyond 3 IQR): Age (day)
WATER_POTABILITY
   - missing cells: ph (15%), Sulfate (24%), Trihalomethanes (5%)
STEEL_FAULTS
   - one value fills much of a column: Steel_Plate_Thickness (37%), Edges_Y_Index (42%)
   - code values (-1, 999, ...): Minimum_of_Luminosity (2%)
   - extreme outliers (> 2 % beyond 3 IQR): Pixels_Areas, X_Perimeter, Y_Perimeter, Sum_of_Luminosity, Maximum_of_Luminosity, Steel_Plate_Thickness, Outside_X_Index, Luminosity_Index
PHONEME
   - 9 exact duplicate rows
   - extreme outliers (> 2 % beyond 3 IQR): V5
MAGIC_TELESCOPE
   - 115 exact duplicate rows
   - ex

### Reading the flags

Not every flag is a problem. Each one either leads to a cleaning rule in step 3 or is kept on purpose:

| Dataset | Flag | Decision |
|---|---|---|
| Concrete | exact copies; identical recipes with a different strength | **rule:** merge each recipe into one row |
| Concrete | one value fills a column | kept: these are real zeros (ingredient not used) and a common test age |
| Water | missing cells | **rule:** training-median fill in step 4 |
| Phoneme, MAGIC, Bank | exact copies | **rule:** removed |
| Bank | `duration` predicts the label alone | **rule:** removed (known only after the call) |
| Bank | `pdays` = −1 in 82 % of rows | **rule:** removed (a code, not a number) |
| Bank | "unknown" in four text columns | kept as its own category |
| Electricity | `date` is a time index | **rule:** removed |
| Electricity | Victoria columns filled with one value | **rule:** filler rows removed |
| Steel | code value in `Minimum_of_Luminosity` | kept: real luminosity readings |
| All | extreme outliers, skewed columns | kept: real measurements, same rule for every dataset |

## 3 · Clean each dataset

Each rule below answers a flag from the audit. A cleaning function returns the features `X`, the label `y` and a short log of what it changed. Exact duplicate rows are removed for every dataset.

### German Credit
Label 1 = good credit risk (raw value 1), 0 = bad (raw value 2). The numeric version of the dataset is already fully coded, and the audit finds no problems, so nothing else changes.

In [11]:
def clean_german(df):
    df = df.drop_duplicates().reset_index(drop=True)
    return df.drop(columns="target"), (df["target"] == 1).astype(int), ["label: 1 = good credit (raw 1)"]

### Concrete
The raw file has exact copies and repeated lab tests: rows with the same mix and age but a slightly different strength. Each group of identical recipes becomes one row with its mean strength. The label is made later, in step 4, from the median strength of the training rows only.

In [5]:
def clean_concrete(df):
    feats = [c for c in df.columns if c != "target"]
    n0, exact, repeats = len(df), int(df.duplicated().sum()), int(df[feats].duplicated().sum() - df.duplicated().sum())
    df = df.groupby(feats, as_index=False, sort=False)["target"].mean()
    log = [f"{exact} exact copies and {repeats} repeated tests merged (mean strength): {n0} -> {len(df)} rows",
           "label: strong vs weak, threshold = training median (step 4)"]
    return df[feats], df["target"], log

### Water Potability
Three columns have real missing values (ph, Sulfate, Trihalomethanes). They are filled in step 4 with the median of the training rows, so no row is dropped and no test information is used.

In [6]:
def clean_water(df):
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    return df.drop(columns="target"), df["target"].astype(int), [f"duplicate rows removed: {n0 - len(df)}",
                                                                  "missing values: training-median fill in step 4"]

### Steel Plates Faults
The label is stored as seven one-hot fault columns; `Other_Faults` becomes the label and all seven are removed from the features (they are the answer). `TypeOfSteel_A300` is the exact opposite of `TypeOfSteel_A400`, so only A400 is kept.

In [7]:
FAULTS = ["Pastry", "Z_Scratch", "K_Scatch", "Stains", "Dirtiness", "Bumps", "Other_Faults"]


def clean_steel(df):
    assert (df[FAULTS].sum(axis=1) == 1).all(), "fault columns are not one-hot"
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    X = df.drop(columns=FAULTS + ["TypeOfSteel_A300"])
    return X, df["Other_Faults"].astype(int), [f"duplicate rows removed: {n0 - len(df)}",
                                               "label: Other_Faults; 7 fault columns and TypeOfSteel_A300 removed"]

### Phoneme and MAGIC Telescope
No cleaning rule is needed beyond removing duplicates. Extreme values are kept: they are real measurements, and the same rule holds for every dataset. The ID column named in the MAGIC documentation is not part of the OpenML file.

### Bank Marketing
- `duration` (length of the call) is only known after the customer has answered, so it leaks the label and is removed.
- `pdays` = −1 is a code for "never contacted", not a number; the same information is in `previous` and `poutcome`, so `pdays` is removed.
- "unknown" is kept as its own category, because it carries information (for example, no earlier campaign).
- Months are coded in calendar order; the other text categories become number codes.

In [8]:
MONTHS = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]


def clean_bank(df):
    df = df.drop(columns=["duration", "pdays"])
    df["month"] = df["month"].astype(str).map({m: i + 1 for i, m in enumerate(MONTHS)})
    cats = [c for c in df.columns if c != "target" and not pd.api.types.is_numeric_dtype(df[c])]
    for c in cats:
        df[c] = df[c].astype(str).astype("category").cat.codes
    return df, ["duration and pdays removed", "month coded 1-12; 'unknown' kept as a category",
                "text categories -> number codes: " + ", ".join(cats)]

### Electricity
- `date` is a running time counter, not a measurement, and is removed.
- In the early part of the data the three Victoria columns were not recorded and hold one constant filler value. Rows where all three hold their filler at the same time are removed.
- `day` (weekday 1–7) is kept as a number in weekday order.

In [9]:
def clean_electricity(df):
    df = df.drop(columns="date")
    fill = {c: df[c].mode().iloc[0] for c in ["vicprice", "vicdemand", "transfer"]}
    is_fill = np.logical_and.reduce([df[c] == v for c, v in fill.items()])
    n0 = len(df)
    df = df[~is_fill].copy()
    df["day"] = df["day"].astype(int)
    return df, ["date removed", f"{n0 - len(df):,} filler rows removed (Victoria not recorded); {len(df):,} rows left",
                "day kept as 1-7"]

### Shared rule for the OpenML datasets
Label 1 = the smaller class. Duplicates are removed after the dataset rule, and every feature must be numeric.

In [12]:
SPECIFIC = {"bank_marketing": clean_bank, "electricity": clean_electricity}


def clean_openml(name, df):
    log = []
    if name in SPECIFIC:
        df, log = SPECIFIC[name](df)
    minority = df["target"].value_counts().idxmin()
    df["target"] = (df["target"] == minority).astype(int)
    assert all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns), f"{name}: non-numeric column left"
    n0 = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    log += [f"label: 1 = '{minority}' (smaller class)", f"duplicate rows removed: {n0 - len(df)}"]
    return df.drop(columns="target").astype(float), df["target"], log


CLEANERS = {"german_credit": clean_german, "concrete": clean_concrete, "water_potability": clean_water,
            "steel_faults": clean_steel, **{n: (lambda df, n=n: clean_openml(n, df)) for n in OPENML_IDS}}

clean = {}
for name in DATASETS:
    X, y, log = CLEANERS[name](raw[name].copy())
    clean[name] = (X, y)
    print(f"{name.upper()}  ->  {len(X):,} rows, {X.shape[1]} features")
    for line in log:
        print("   -", line)

GERMAN_CREDIT  ->  1,000 rows, 24 features
   - label: 1 = good credit (raw 1)
CONCRETE  ->  992 rows, 8 features
   - 25 exact copies and 13 repeated tests merged (mean strength): 1030 -> 992 rows
   - label: strong vs weak, threshold = training median (step 4)
WATER_POTABILITY  ->  3,276 rows, 9 features
   - duplicate rows removed: 0
   - missing values: training-median fill in step 4
STEEL_FAULTS  ->  1,941 rows, 26 features
   - duplicate rows removed: 0
   - label: Other_Faults; 7 fault columns and TypeOfSteel_A300 removed
PHONEME  ->  5,395 rows, 5 features
   - label: 1 = '2' (smaller class)
   - duplicate rows removed: 9
MAGIC_TELESCOPE  ->  18,905 rows, 10 features
   - label: 1 = 'h' (smaller class)
   - duplicate rows removed: 115
BANK_MARKETING  ->  45,195 rows, 14 features
   - duration and pdays removed
   - month coded 1-12; 'unknown' kept as a category
   - text categories -> number codes: job, marital, education, default, housing, loan, contact, poutcome
   - label: 1

## 4 · Prepare: one pipeline for all datasets

1. **Sample** 350 rows, keeping the class balance.
2. **Split** them into 280 training and 70 test rows, again keeping the balance. From here on, everything is fitted on the training rows only.
3. **Fill** missing values with the training median (Water only).
4. **Drop** constant features, then one feature of every pair correlated above 0.85.
5. **Select** the 14 most useful features by cross-validated permutation importance — only when more than 14 remain (German, Steel).
6. **Scale** every feature to [0, π], the angle range of the quantum encoding. Test values are not clipped, so they may fall slightly outside.

For Concrete, the balance in steps 1–2 uses a temporary label (above or below the median of all rows); the real label is then set from the median strength of the training rows.

In [13]:
def prepare(name, X, y):
    continuous = name == "concrete"
    strat = (y >= y.median()).astype(int) if continuous else y
    sub, _ = train_test_split(np.arange(len(X)), train_size=N_ROWS, stratify=strat, random_state=SEED)
    tr, te = train_test_split(sub, test_size=TEST_SHARE, stratify=strat.iloc[sub], random_state=SEED)
    X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
    if continuous:
        threshold = y_tr.median()
        y_tr, y_te = (y_tr >= threshold).astype(int), (y_te >= threshold).astype(int)
    info = {}

    if X_tr.isna().any().any():
        imputer = SimpleImputer(strategy="median").fit(X_tr)
        X_tr = pd.DataFrame(imputer.transform(X_tr), columns=X.columns, index=X_tr.index)
        X_te = pd.DataFrame(imputer.transform(X_te), columns=X.columns, index=X_te.index)
        info["filled"] = "training median"

    constant = [c for c in X_tr.columns if X_tr[c].nunique() <= 1]
    upper = X_tr.drop(columns=constant).corr().abs()
    upper = upper.where(np.triu(np.ones(upper.shape), k=1).astype(bool))
    correlated = [c for c in upper.columns if any(upper[c] > CORR_LIMIT)]
    X_tr, X_te = X_tr.drop(columns=constant + correlated), X_te.drop(columns=constant + correlated)
    info["dropped"] = ", ".join(constant + correlated) or "-"

    if X_tr.shape[1] > QUBIT_CAP:
        scores = pd.DataFrame(index=X_tr.columns)
        for k, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_tr, y_tr)):
            rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X_tr.iloc[a], y_tr.iloc[a])
            scores[k] = permutation_importance(rf, X_tr.iloc[b], y_tr.iloc[b], n_repeats=10, random_state=SEED,
                                               scoring="f1", n_jobs=-1).importances_mean
        keep = scores.mean(axis=1).sort_values(ascending=False).index[:QUBIT_CAP].tolist()
        info["selected"] = f"top {QUBIT_CAP} of {X_tr.shape[1]}"
        X_tr, X_te = X_tr[keep], X_te[keep]

    scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(X_tr)
    train = pd.DataFrame(scaler.transform(X_tr), columns=X_tr.columns).assign(target=y_tr.values)
    test = pd.DataFrame(scaler.transform(X_te), columns=X_tr.columns).assign(target=y_te.values)
    return train, test, info


ready, summary = {}, {}
for name in DATASETS:
    train, test, info = prepare(name, *clean[name])
    ready[name] = (train, test)
    summary[name] = {"qubits": train.shape[1] - 1, "train rows": len(train), "test rows": len(test),
                     "label = 1 (train)": f"{train.target.mean():.0%}",
                     "test values outside [0, π]": f"{((test.iloc[:, :-1] < 0) | (test.iloc[:, :-1] > np.pi)).values.mean():.1%}",
                     "filled": info.get("filled", "-"), "dropped (train only)": info["dropped"],
                     "selected": info.get("selected", "-")}

pd.DataFrame(summary).T

,qubits,train rows,test rows,label = 1 (train),"test values outside [0, π]",filled,dropped (train only),selected
german_credit,14,280,70,70%,0.0%,-,-,top 14 of 24
concrete,8,280,70,50%,0.4%,-,-,-
water_potability,9,280,70,39%,0.8%,training median,-,-
steel_faults,14,280,70,35%,0.5%,-,"X_Maximum, Y_Maximum, X_Perimeter, Y_Perimeter...",top 14 of 16
phoneme,5,280,70,29%,0.9%,-,-,-
magic_telescope,8,280,70,35%,1.6%,-,"fConc, fConc1",-
bank_marketing,14,280,70,12%,0.1%,-,-,-
electricity,6,280,70,42%,0.7%,-,vicprice,-


## 5 · Check and save

Each prepared dataset is compared value by value with the file of the same name in `data/processed/`. **MATCH** means the notebook rebuilds exactly the data used in all experiments.

In [14]:
check = {}
for name in DATASETS:
    row = {}
    for part, new in zip(["train", "test"], ready[name]):
        path = PROCESSED / f"{name}_{part}_ready.csv"
        if not path.exists():
            row[part] = "no file yet"
            continue
        old = pd.read_csv(path, float_precision="round_trip")
        same = list(old.columns) == list(new.columns) and old.shape == new.shape and np.array_equal(old.values, new.values)
        row[part] = "MATCH" if same else "DIFFERENT"
    check[name] = row

pd.DataFrame(check).T

,train,test
german_credit,MATCH,MATCH
concrete,MATCH,MATCH
water_potability,MATCH,MATCH
steel_faults,MATCH,MATCH
phoneme,MATCH,MATCH
magic_telescope,MATCH,MATCH
bank_marketing,MATCH,MATCH
electricity,MATCH,MATCH


In [ ]:
if SAVE:
    for name in DATASETS:
        for part, df in zip(["train", "test"], ready[name]):
            df.to_csv(PROCESSED / f"{name}_{part}_ready.csv", index=False)
    print(f"Saved {2 * len(DATASETS)} files to {PROCESSED}")
else:
    print("SAVE = False: nothing was written.")

SAVE = False: nothing was written.


: 